# Yahoo Finance Historical Stock Machine Learning Pipeline

This notebook pulls **30 years of daily and weekly historical stock data** from Yahoo Finance for a basket of representative assets (`MU`, `GOOG`, `TSLA`, `SPY`). It calculates standard technical indicators (SMAs, EMAs, Bollinger Bands, RSI, MACD, etc.), sets up chronological data splits, and evaluates machine learning models (Logistic Regression and Random Forest) to determine if a larger historical sample size improves predictive performance compared to smaller datasets.

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

RANDOM_STATE = 42
MULTI_SYMBOL = ["MU", "GOOG", "TSLA", "SPY"]


## 1. Feature Engineering Class

The `feature_create` class groups indicators by symbol to prevent cross-asset leakage and dynamically adjusts the lookback periods depending on the timeframe (daily vs. weekly) to prevent excessive warmup data loss.

In [2]:
class feature_create:
    '''Class to help incorporate all methods for 
    metric creation. Computes features grouped by symbol to prevent data leakage
    and dynamically scales lookback windows depending on the timeframe.'''

    def __init__(self, df, timeframe="daily"):
        self.df = df
        self.timeframe = timeframe.lower()
    
    def calculate_moving_avg(self):
        # Set windows based on timeframe (weekly is scaled down to prevent massive warmup data loss)
        if self.timeframe == "weekly":
            windows = (5, 10, 20, 30)
        else:
            windows = (10, 20, 50, 100, 200)
            
        # Group by symbol before rolling or ewm calculations to prevent cross-symbol bleeding
        for w in windows:
            self.df[f'SMA_{w}'] = self.df.groupby('symbol')['close'].transform(lambda x: x.rolling(window=w).mean())
            self.df[f'ewm_{w}'] = self.df.groupby('symbol')['close'].transform(lambda x: x.ewm(span=w, adjust=False).mean())
            self.df[f'price_to_ma{w}'] = self.df['close'] / self.df[f'SMA_{w}'] - 1
            
        # Moving average crossover
        if self.timeframe == "weekly":
            self.df['ma_cross'] = self.df['SMA_10'] / self.df['SMA_30'] - 1
        else:
            self.df['ma_cross'] = self.df['SMA_50'] / self.df['SMA_200'] - 1
        return self

    def bollinger(self):
        self.df['BB_Mid'] = self.df.groupby('symbol')['close'].transform(lambda x: x.rolling(window=20).mean())
        bb_std = self.df.groupby('symbol')['close'].transform(lambda x: x.rolling(window=20).std())
        self.df['BB_Upper'] = self.df['BB_Mid'] + 2 * bb_std
        self.df['BB_Lower'] = self.df['BB_Mid'] - 2 * bb_std
        
        # Bollinger Band Percentile (bb_pct - where price sits in the band: 0 = bottom, 1 = top)
        denom = self.df['BB_Upper'] - self.df['BB_Lower']
        self.df['bb_pct'] = np.where(denom == 0, 0.5, (self.df['close'] - self.df['BB_Lower']) / denom)
        return self

    def RSI(self):
        def compute_rsi(series):
            delta = series.diff()
            gain = delta.clip(lower=0)
            loss = -delta.clip(upper=0)
            avg_gain = gain.ewm(alpha=1/14, min_periods=14, adjust=False).mean()
            avg_loss = loss.ewm(alpha=1/14, min_periods=14, adjust=False).mean()
            rs = avg_gain / avg_loss
            return 100 - (100 / (1 + rs))
            
        self.df['RSI'] = self.df.groupby('symbol')['close'].transform(compute_rsi)
        return self

    def MACD(self):
        def compute_macd_line(series):
            return series.ewm(span=12, adjust=False).mean() - series.ewm(span=26, adjust=False).mean()
        def compute_macd_signal(macd_series):
            return macd_series.ewm(span=9, adjust=False).mean()
            
        macd_line = self.df.groupby('symbol')['close'].transform(compute_macd_line)
        self.df['MACD'] = macd_line / self.df['close']
        self.df['MACD_Signal'] = self.df.groupby('symbol')['MACD'].transform(compute_macd_signal)
        self.df['MACD_Hist'] = self.df['MACD'] - self.df['MACD_Signal']
        return self

    def ret(self):
        def ret_1(series):
            return series.pct_change()
        def log_ret(series):
            return np.log(series).diff()

        self.df['ret_1'] = self.df.groupby('symbol')['close'].transform(ret_1)
        self.df['log_ret'] = self.df.groupby('symbol')['close'].transform(log_ret)
        
        for lag in (1, 2, 3):
            self.df[f"ret_lag{lag}"] = self.df.groupby('symbol')["ret_1"].transform(lambda x: x.shift(lag))
        return self

    def momentum(self):
        if self.timeframe == "weekly":
            horizons = (2, 4, 8, 12, 26)
        else:
            horizons = (10, 20, 30, 50, 100, 200)
            
        for k in horizons:
            self.df[f"mom_{k}"] = self.df.groupby('symbol')['close'].transform(lambda x: x / x.shift(k) - 1)
        return self

    def volatility_and_volume(self):
        vol_w = 10 if self.timeframe == "weekly" else 20
        self.df[f'volatility_{vol_w}'] = self.df.groupby('symbol')['ret_1'].transform(lambda x: x.rolling(window=vol_w).std())
        
        vol_windows = (4, 10, 26) if self.timeframe == "weekly" else (10, 20, 50)
        for w in vol_windows:
            vol_avg = self.df.groupby('symbol')['volume'].transform(lambda x: x.rolling(window=w).mean())
            self.df[f'vol_ratio{w}'] = self.df['volume'] / vol_avg
        return self

    def advanced_indicators(self):
        self.df['range_hl'] = (self.df['high'] - self.df['low']) / self.df['close']
        
        prev_close = self.df.groupby('symbol')['close'].shift(1)
        self.df['gap'] = (self.df['open'] - prev_close) / prev_close
        
        tr1 = self.df['high'] - self.df['low']
        tr2 = (self.df['high'] - prev_close).abs()
        tr3 = (self.df['low'] - prev_close).abs()
        tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
        
        alpha = 1/10 if self.timeframe == "weekly" else 1/14
        min_periods = 10 if self.timeframe == "weekly" else 14
        atr_raw = tr.groupby(self.df['symbol']).transform(lambda x: x.ewm(alpha=alpha, min_periods=min_periods, adjust=False).mean())
        self.df[f'atr_{int(1/alpha)}'] = atr_raw / self.df['close']
        
        up_move = self.df.groupby('symbol')['high'].diff()
        down_move = -self.df.groupby('symbol')['low'].diff()
        plus_dm = pd.Series(np.where((up_move > down_move) & (up_move > 0), up_move, 0.0), index=self.df.index)
        minus_dm = pd.Series(np.where((down_move > up_move) & (down_move > 0), down_move, 0.0), index=self.df.index)
        
        plus_dm_smoothed = plus_dm.groupby(self.df['symbol']).transform(lambda x: x.ewm(alpha=alpha, min_periods=min_periods, adjust=False).mean())
        minus_dm_smoothed = minus_dm.groupby(self.df['symbol']).transform(lambda x: x.ewm(alpha=alpha, min_periods=min_periods, adjust=False).mean())
        
        plus_di = 100 * plus_dm_smoothed / atr_raw
        minus_di = 100 * minus_dm_smoothed / atr_raw
        dx = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di)
        self.df[f'adx_{int(1/alpha)}'] = dx.groupby(self.df['symbol']).transform(lambda x: x.ewm(alpha=alpha, min_periods=min_periods, adjust=False).mean())
        
        typical = (self.df['high'] + self.df['low'] + self.df['close']) / 3
        money_flow = typical * self.df['volume']
        typical_shift = typical.groupby(self.df['symbol']).shift(1)
        
        pos_flow = pd.Series(np.where(typical > typical_shift, money_flow, 0.0), index=self.df.index)
        neg_flow = pd.Series(np.where(typical < typical_shift, money_flow, 0.0), index=self.df.index)
        
        mfi_w = 10 if self.timeframe == "weekly" else 14
        pos_mf = pos_flow.groupby(self.df['symbol']).transform(lambda x: x.rolling(mfi_w).sum())
        neg_mf = neg_flow.groupby(self.df['symbol']).transform(lambda x: x.rolling(mfi_w).sum())
        self.df[f'mfi_{mfi_w}'] = 100 - (100 / (1 + pos_mf / neg_mf))
        
        denom = self.df['high'] - self.df['low']
        denom = np.where(denom == 0, 1e-9, denom)
        mfm = ((self.df['close'] - self.df['low']) - (self.df['high'] - self.df['close'])) / denom
        mfv = mfm * self.df['volume']
        
        cmf_w = 10 if self.timeframe == "weekly" else 20
        num_sum = mfv.groupby(self.df['symbol']).transform(lambda x: x.rolling(cmf_w).sum())
        denom_sum = self.df['volume'].groupby(self.df['symbol']).transform(lambda x: x.rolling(cmf_w).sum())
        self.df[f'cmf_{cmf_w}'] = num_sum / denom_sum
        return self

    def create_all_features(self):
        self.calculate_moving_avg()
        self.bollinger()
        self.RSI()
        self.MACD()
        self.ret()
        self.momentum()
        self.volatility_and_volume()
        self.advanced_indicators()
        return self

## 2. Chronological Splitting and Labeling

This function cleans features, sorts chronologically, and establishes the 5 target classes strictly from training-set quantiles to prevent target leakage.

In [3]:
def preprocess_and_split(table, zero_thresh=0.02):
    # Dynamically select features based on timeframe
    ma_cols = ["price_to_ma10", "price_to_ma50", "price_to_ma200"] if "price_to_ma50" in table.columns else ["price_to_ma5", "price_to_ma20"]
    vol_col = ["vol_ratio20"] if "vol_ratio20" in table.columns else ["vol_ratio10"]

    FEATURE_COLS = [
        "ret_1", "ret_lag1", "ret_lag2", "ret_lag3",
        "ma_cross", "RSI", "MACD_Hist", "range_hl", "gap",
        "volatility_10" if "volatility_10" in table.columns else "volatility_20",
        "adx_10" if "adx_10" in table.columns else "adx_14"
    ] + ma_cols + vol_col

    # Clean data (replace infs with NaN and drop NaNs)
    table = table.replace([np.inf, -np.inf], np.nan).dropna(subset=FEATURE_COLS + ["Next_Return"]).copy()
    
    # Sort chronologically by timestamp
    table['timestamp'] = pd.to_datetime(table['timestamp'])
    table = table.sort_values('timestamp')
    
    unique_dates = sorted(table['timestamp'].unique())
    N = len(unique_dates)
    
    TRAIN_FRAC = 0.70
    VAL_FRAC   = 0.15
    
    TRAIN_END = unique_dates[int(N * TRAIN_FRAC)]
    VAL_END   = unique_dates[int(N * (TRAIN_FRAC + VAL_FRAC))]
    
    train_df = table[table['timestamp'] <= TRAIN_END].copy()
    val_df   = table[(table['timestamp'] > TRAIN_END) & (table['timestamp'] <= VAL_END)].copy()
    test_df  = table[table['timestamp'] > VAL_END].copy()

    def classify_return(r, q1, q3, thresh):
        if r < q1:
            return 0  # Strong Down
        elif r < -thresh:
            return 1  # Moderate Down
        elif r <= thresh:
            return 2  # Flat
        elif r <= q3:
            return 3  # Moderate Up
        else:
            return 4  # Strong Up

    symbols = table['symbol'].unique() if 'symbol' in table.columns else [None]
    
    for df in [train_df, val_df, test_df]:
        df['Target'] = np.nan

    for sym in symbols:
        if sym is not None:
            train_sym = train_df[train_df['symbol'] == sym]
        else:
            train_sym = train_df
            
        clean_returns = train_sym['Next_Return'].dropna()
        if len(clean_returns) == 0:
            continue
            
        q1 = clean_returns.quantile(0.25)
        q3 = clean_returns.quantile(0.75)
        avg_movement = clean_returns.abs().mean()
        thresh = zero_thresh * avg_movement
        
        if sym is not None:
            train_df.loc[train_df['symbol'] == sym, 'Target'] = train_df.loc[train_df['symbol'] == sym, 'Next_Return'].apply(
                lambda x: classify_return(x, q1, q3, thresh) if pd.notna(x) else np.nan
            )
            val_df.loc[val_df['symbol'] == sym, 'Target'] = val_df.loc[val_df['symbol'] == sym, 'Next_Return'].apply(
                lambda x: classify_return(x, q1, q3, thresh) if pd.notna(x) else np.nan
            )
            test_df.loc[test_df['symbol'] == sym, 'Target'] = test_df.loc[test_df['symbol'] == sym, 'Next_Return'].apply(
                lambda x: classify_return(x, q1, q3, thresh) if pd.notna(x) else np.nan
            )
        else:
            train_df['Target'] = train_df['Next_Return'].apply(
                lambda x: classify_return(x, q1, q3, thresh) if pd.notna(x) else np.nan
            )
            val_df['Target'] = val_df['Next_Return'].apply(
                lambda x: classify_return(x, q1, q3, thresh) if pd.notna(x) else np.nan
            )
            test_df['Target'] = test_df['Next_Return'].apply(
                lambda x: classify_return(x, q1, q3, thresh) if pd.notna(x) else np.nan
            )

    train_df = train_df.dropna(subset=['Target'])
    val_df = val_df.dropna(subset=['Target'])
    test_df = test_df.dropna(subset=['Target'])
    
    train_df['Target'] = train_df['Target'].astype(int)
    val_df['Target'] = val_df['Target'].astype(int)
    test_df['Target'] = test_df['Target'].astype(int)

    X_train = train_df[FEATURE_COLS].to_numpy()
    Y_train = train_df['Target'].to_numpy()
    
    X_val = val_df[FEATURE_COLS].to_numpy()
    Y_val = val_df['Target'].to_numpy()
    
    X_test = test_df[FEATURE_COLS].to_numpy()
    Y_test = test_df['Target'].to_numpy()

    print('Size of X_train', X_train.shape)
    print('Size of y_train', Y_train.shape)
    print('Size of X_val', X_val.shape)
    print('Size of y_val', Y_val.shape)
    print('Size of X_test', X_test.shape)
    print('Size of y_test', Y_test.shape)

    return X_train, Y_train, X_val, Y_val, X_test, Y_test

## 3. Data Download & Processing Function

This pulls 30 years of daily and weekly data, stacks/reshapes it into a long format, sorts it chronologically, and appends the next period's return.

In [4]:
def download_and_reshape(timeframe="daily", start_date="1996-01-01", end_date="2026-07-01"):
    interval = "1d" if timeframe.lower() == "daily" else "1wk"
    print(f"Downloading {timeframe} data from {start_date} to {end_date}...")
    
    data = yf.download(tickers=MULTI_SYMBOL, start=start_date, end=end_date, interval=interval)
    
    # Reshape from wide to long format
    data_long = data.stack(level='Ticker').reset_index()
    
    # Rename columns to standard conventions
    data_long = data_long.rename(columns={ 
        'Date': 'timestamp',
        'Ticker': 'symbol',
        'Open': 'open',
        'High': 'high',
        'Low': 'low',
        'Close': 'close',
        'Volume': 'volume'
    })
    
    # Sort chronologically per symbol and calculate Target returns
    data_long = data_long.sort_values(by=['symbol', 'timestamp']).reset_index(drop=True)
    data_long['Next_Return'] = data_long.groupby('symbol')['close'].transform(lambda x: x.pct_change().shift(-1))
    
    return data_long

## 4. Run Data Pull & Feature Engineering

In [5]:
print("=== DAILY DATASET ===")
daily_raw = download_and_reshape("daily")
daily_features = feature_create(daily_raw, timeframe="daily").create_all_features().df
daily_features = daily_features.dropna()

print("\n=== WEEKLY DATASET ===")
weekly_raw = download_and_reshape("weekly")
weekly_features = feature_create(weekly_raw, timeframe="weekly").create_all_features().df
weekly_features = weekly_features.dropna()

=== DAILY DATASET ===

=== WEEKLY DATASET ===


[*********************100%***********************]  4 of 4 completed
[*********************100%***********************]  4 of 4 completed


## 5. Model Evaluation Pipeline

This mirrors the exact models used on the Alpaca dataset: Dummy (Baseline), Multinomial Logistic Regression, and Random Forest.

In [6]:
def evaluate_timeframe(df, label):
    print(f"\n=======================================")
    print(f"Evaluating Timeframe: {label.upper()}")
    print(f"=======================================")
    
    # Split data
    X_train, Y_train, X_val, Y_val, X_test, Y_test = preprocess_and_split(df)
    
    # Standardize features using training fit
    scaler = StandardScaler()
    X_train_std = scaler.fit_transform(X_train)
    X_val_std = scaler.transform(X_val)
    X_test_std = scaler.transform(X_test)
    
    # Model 1: Dummy Baseline
    majority = DummyClassifier(strategy="most_frequent").fit(X_train_std, Y_train)
    maj_pred = majority.predict(X_test_std)
    
    # Model 2: Multinomial Logistic Regression
    logreg = LogisticRegression(max_iter=2000, C=1.0, random_state=RANDOM_STATE)
    logreg.fit(X_train_std, Y_train)
    log_pred = logreg.predict(X_test_std)
    
    # Model 3: Random Forest
    rf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=RANDOM_STATE)
    rf.fit(X_train, Y_train)  # Standardized features used consistently
    rf_pred = rf.predict(X_test)
    
    # Print Test Results
    print(f"\nTest-Set Results ({label.upper()}):")
    print(f"  Majority class : acc={accuracy_score(Y_test, maj_pred):.4f}  macroF1={f1_score(Y_test, maj_pred, average='macro', zero_division=0):.4f}")
    print(f"  Logistic reg   : acc={accuracy_score(Y_test, log_pred):.4f}  macroF1={f1_score(Y_test, log_pred, average='macro', zero_division=0):.4f}")
    print(f"  Random Forest  : acc={accuracy_score(Y_test, rf_pred):.4f}  macroF1={f1_score(Y_test, rf_pred, average='macro', zero_division=0):.4f}")
    
    LABELS = ["Strong Down", "Moderate Down", "Flat", "Moderate Up", "Strong Up"]
    print("\nPer-class report (logistic regression):")
    print(classification_report(Y_test, log_pred, labels=[0, 1, 2, 3, 4], target_names=LABELS, digits=3, zero_division=0))
    print("\nPer-class report (random forest):")
    print(classification_report(Y_test, rf_pred, labels=[0, 1, 2, 3, 4], target_names=LABELS, digits=3, zero_division=0))

## 6. Run Evaluations

In [8]:
evaluate_timeframe(daily_features, "daily")
evaluate_timeframe(weekly_features, "weekly")


Evaluating Timeframe: DAILY
Size of X_train (15103, 15)
Size of y_train (15103,)
Size of X_val (4484, 15)
Size of y_val (4484,)
Size of X_test (4480, 15)
Size of y_test (4480,)

Test-Set Results (DAILY):
  Majority class : acc=0.2417  macroF1=0.0779
  Logistic reg   : acc=0.3029  macroF1=0.2112
  Random Forest  : acc=0.3096  macroF1=0.2105

Per-class report (logistic regression):
               precision    recall  f1-score   support

  Strong Down      0.315     0.270     0.291      1243
Moderate Down      0.300     0.035     0.063       855
         Flat      0.000     0.000     0.000        62
  Moderate Up      0.289     0.500     0.367      1083
    Strong Up      0.312     0.363     0.335      1237

     accuracy                          0.303      4480
    macro avg      0.243     0.234     0.211      4480
 weighted avg      0.301     0.303     0.274      4480


Per-class report (random forest):
               precision    recall  f1-score   support

  Strong Down      0.316   

## Revisitng the LSTM model
-- Applying the LSTM deep learning model to our case - based on this paper: https://www.sciencedirect.com/science/article/abs/pii/S0377221717310652?via%3Dihub

-  Our current implementation treats the LSTM as a standard classifier on top of hand-crafted features, which defeats the purpose of the LSTM architecture and causes it to overfit to market noise.

-  LSTM acts as a feature extractor, learning temporal representations of returns directly. Feeding pre-calculated indicators over short sequence windows creates a high-dimensional, highly correlated feature space. The LSTM gets overwhelmed by noise and suffers from the curse of dimensionality, causing it to overfit and perform at near-random chance.

-  B. Sequence Length (Lookback Window)

What the paper did (Section 3.2.1): Used a sequence length of **240** trading days (approximately one full year of market data).
Used an LSTM_WINDOW of **20** trading days (approximately 4 weeks). 20 days is too short for an LSTM to learn long-term temporal relationships or volatility cycles. LSTMs need longer sequences (like the paper's 240 days) to leverage their recurrent "memory gates" effectively.

- C. Absolute 5-Class Target vs. Relative Cross-Sectional Binary Target

What the paper did (Section 3.2.2): Used a binary classification target based on relative performance. Stocks were split into two classes (0 and 1) based on whether their next-day return outperformed or underperformed the cross-sectional median of the entire S&P 500 on that day.

What the notebooks did: Used an absolute 5-class target (Strong Down to Strong Up based on absolute return quantiles).

Why it matters: 5-class classification is significantly harder to optimize than a binary split, especially with severe class imbalance (like the "Flat" class). Furthermore, classifying relative cross-sectional performance (as in the paper) factors out market-wide beta shifts, allowing the LSTM to focus purely on relative stock dynamics.

-  D. Over-parameterized Architecture

What the paper did (Section 3.3): Used a very simple single-layer LSTM with only 25 hidden units, a dropout of 0.1, and an RMSprop optimizer.
What the notebooks did: Used a stacked multilayer LSTM (LSTM(64) -> LSTM(32) -> Dense(32) -> Dense(5)) with recurrent dropout and an Adam optimizer.
Why it matters: A deeper network trained on short sequences (20 days) and high-dimensional features (29+) has too many degrees of freedom, leading to immediate overfitting on training noise.


2. As a baseline, we should first implement the LSTM model as it was implemented in the paper to test how well the model works for our data. Once we know how the model behaves, we can tweak it to see if we can further improve on the performance or predict more complex labels.

In [9]:
def preprocess_split_LSTM(table):
    # Sort chronologically
    table = table.sort_values(['symbol', 'timestamp']).reset_index(drop=True)
    
    # 1. Calculate daily returns (m = 1)
    table['Return'] = table.groupby('symbol')['close'].pct_change()
    table = table.dropna(subset=['Return']).copy()
    
    # 2. Calculate next day's return (Target Source)
    table['Next_Return'] = table.groupby('symbol')['Return'].shift(-1)
    table = table.dropna(subset=['Next_Return']).copy()
    
    # 3. Calculate cross-sectional median return per timestamp
    table['threshold'] = table.groupby('timestamp')['Next_Return'].transform('median')
    
    # 4. Binary target: 1 if next return >= daily median, 0 otherwise
    table['Target'] = np.where(table['Next_Return'] >= table['threshold'], 1, 0)
    
    # Chronological splitting
    unique_dates = sorted(table['timestamp'].unique())
    N = len(unique_dates)
    
    TRAIN_FRAC = 0.70
    VAL_FRAC   = 0.15
    
    TRAIN_END = unique_dates[int(N * TRAIN_FRAC)]
    VAL_END   = unique_dates[int(N * (TRAIN_FRAC + VAL_FRAC))]
    
    # Compute training stats for standardization
    train_subset = table[table['timestamp'] <= TRAIN_END]
    train_stats = train_subset.groupby('symbol')['Return'].agg(['mean', 'std']).reset_index()
    train_stats.rename(columns={'mean': 'mu_train', 'std': 'sigma_train'}, inplace=True)
    
    # Merge training stats back BEFORE splitting
    table = table.merge(train_stats, on='symbol', how='left')
    
    # Split data
    train_df = table[table['timestamp'] <= TRAIN_END].copy()
    val_df   = table[(table['timestamp'] > TRAIN_END) & (table['timestamp'] <= VAL_END)].copy()
    test_df  = table[table['timestamp'] > VAL_END].copy()
    
    # Standardize returns using training statistics
    train_df['Std_Return'] = (train_df['Return'] - train_df['mu_train']) / train_df['sigma_train']
    val_df['Std_Return']   = (val_df['Return'] - val_df['mu_train']) / val_df['sigma_train']
    test_df['Std_Return']  = (test_df['Return'] - test_df['mu_train']) / test_df['sigma_train']

    return train_df, val_df, test_df

In [10]:
# Helper to create sliding windows of shape (samples, timesteps, features)
def create_lstm_sequences(df_split, window_size=240):
    sequences = []
    targets = []
    
    # We group by symbol to avoid mixing data between different stocks
    for symbol, group in df_split.groupby('symbol'):
        values = group['Std_Return'].values
        labels = group['Target'].values # 1 or 0 binary target
        
        for i in range(window_size, len(values)):
            sequences.append(values[i-window_size:i]) # past 240 standardized returns
            targets.append(labels[i])                 # corresponding target
            
    # Reshape sequences to (samples, 240, 1) for the LSTM
    X = np.expand_dims(np.array(sequences), axis=-1)
    y = np.array(targets)
    return X, y



In [11]:
# Apply to each split
train_df, val_df, test_df = preprocess_split_LSTM(daily_features)

X_train, y_train = create_lstm_sequences(train_df)
X_val, y_val = create_lstm_sequences(val_df)
X_test, y_test = create_lstm_sequences(test_df)
test_df.columns

In [12]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

RANDOM_STATE = 42
tf.keras.utils.set_random_seed(RANDOM_STATE)

# 1. Flatten the 3D LSTM inputs to 2D for the benchmark models
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_val_flat = X_val.reshape(X_val.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

print(f"LSTM Input Shape: {X_train.shape}")
print(f"Benchmark Model Input Shape: {X_train_flat.shape}")

# 2. Train Benchmark Models on the Binary Targets
# Dummy Classifier
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train_flat, y_train)
dummy_pred = dummy.predict(X_test_flat)

# Logistic Regression
lr = LogisticRegression(max_iter=1000, C=1.0, random_state=RANDOM_STATE)
lr.fit(X_train_flat, y_train)
lr_pred = lr.predict(X_test_flat)

# Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=RANDOM_STATE)
rf.fit(X_train_flat, y_train)
rf_pred = rf.predict(X_test_flat)

# 3. Build and Train the LSTM model exactly as described in the paper
lstm_model = Sequential([
    Input(shape=(X_train.shape[1], X_train.shape[2])), # (240, 1)
    LSTM(25, dropout=0.1, recurrent_dropout=0.1),
    Dense(2, activation="softmax")
])

lstm_model.compile(
    optimizer="rmsprop",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

print("\nTraining LSTM model...")
history = lstm_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=128,
    callbacks=[early_stopping],
    verbose=1
)

# LSTM Predictions
lstm_probs = lstm_model.predict(X_test)
lstm_pred = np.argmax(lstm_probs, axis=1)


LSTM Input Shape: (14139, 240, 1)
Benchmark Model Input Shape: (14139, 240)

Training LSTM model...
Epoch 1/50
111/111 ━━━━━━━━━━━━━━━━━━━━ 4s 31ms/step - accuracy: 0.5499 - loss: 0.6887 - val_accuracy: 0.5000 - val_loss: 0.6980
Epoch 2/50
111/111 ━━━━━━━━━━━━━━━━━━━━ 3s 30ms/step - accuracy: 0.5502 - loss: 0.6882 - val_accuracy: 0.4991 - val_loss: 0.6979
Epoch 3/50
111/111 ━━━━━━━━━━━━━━━━━━━━ 3s 30ms/step - accuracy: 0.5506 - loss: 0.6881 - val_accuracy: 0.4997 - val_loss: 0.6981
Epoch 4/50
111/111 ━━━━━━━━━━━━━━━━━━━━ 3s 31ms/step - accuracy: 0.5505 - loss: 0.6882 - val_accuracy: 0.4997 - val_loss: 0.6982
Epoch 5/50
111/111 ━━━━━━━━━━━━━━━━━━━━ 3s 31ms/step - accuracy: 0.5508 - loss: 0.6881 - val_accuracy: 0.4991 - val_loss: 0.6982
Epoch 6/50
111/111 ━━━━━━━━━━━━━━━━━━━━ 3s 31ms/step - accuracy: 0.5503 - loss: 0.6881 - val_accuracy: 0.4997 - val_loss: 0.6982
Epoch 7/50
111/111 ━━━━━━━━━━━━━━━━━━━━ 3s 31ms/step - accuracy: 0.5509 - loss: 0.6879 - val_accuracy: 0.4994 - val_loss: 0.69

In [13]:
# Evaluate all models
print("\n=============================================")
print("MODEL COMPARISON (BINARY CROSS-SECTIONAL TARGET)")
print("=============================================")

models = {
    "Majority Class Baseline": dummy_pred,
    "Logistic Regression": lr_pred,
    "Random Forest": rf_pred,
    "LSTM Classifier": lstm_pred
}

for name, preds in models.items():
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average="macro", zero_division=0)
    print(f"{name:<25} : Accuracy = {acc:.4f} | Macro F1 = {f1:.4f}")
    
# Print detailed classification report for the LSTM
print("\nDetailed Per-Class Report (LSTM):")
print(classification_report(y_test, lstm_pred, target_names=["Underperform", "Outperform"], digits=3, zero_division=0))



MODEL COMPARISON (BINARY CROSS-SECTIONAL TARGET)
Majority Class Baseline   : Accuracy = 0.5000 | Macro F1 = 0.3333
Logistic Regression       : Accuracy = 0.4920 | Macro F1 = 0.4232
Random Forest             : Accuracy = 0.5000 | Macro F1 = 0.3333
LSTM Classifier           : Accuracy = 0.5006 | Macro F1 = 0.3371

Detailed Per-Class Report (LSTM):
              precision    recall  f1-score   support

Underperform      0.583     0.004     0.008      1760
  Outperform      0.500     0.997     0.666      1760

    accuracy                          0.501      3520
   macro avg      0.542     0.501     0.337      3520
weighted avg      0.542     0.501     0.337      3520



In [14]:
lstm_probs.shape

In [15]:
outperform_probs = lstm_probs[:,1]
underperform_probs = lstm_probs[:,0]

# 1. Sort the test set in the exact same order as sequences (symbol first, then date)
test_results = test_df.sort_values(by=['symbol', 'timestamp']).copy()

# 2. Skip the first 240 rows per symbol group using cumcount (robust across pandas versions)
mask = test_results.groupby('symbol').cumcount() >= 240
test_results = test_results[mask].copy()

# 3. Reset the index
test_results = test_results.reset_index(drop=True)

# 4. Now the length of test_results matches the length of outperform_probabilities exactly
test_results['Outperform_Prob'] = outperform_probs
test_results['Underperform_Prob'] = underperform_probs

def get_daily_portfolio(group, k=1):
    # Function to get top/bottom k stocks per day
    group.sort_values(by='Outperform_Prob', ascending=False, inplace=True)
    # Get first 10
    long_leg = group['symbol'].iloc[:k].to_list()
    # Get last 10
    short_leg = group['symbol'].iloc[-k:].to_list()
    
    return pd.Series({"Stocks to buy": long_leg, "Stocks to sell": short_leg})

# 5. Group by timestamp to generate the portfolio for every day in the test set
daily_portfolios = test_results.groupby('timestamp').apply(lambda g: get_daily_portfolio(g, k=1))
print(daily_portfolios.head(20))


           Stocks to buy Stocks to sell
timestamp                              
2022-12-21        [GOOG]           [MU]
2022-12-22        [TSLA]           [MU]
2022-12-23        [GOOG]         [TSLA]
2022-12-27        [TSLA]           [MU]
2022-12-28        [GOOG]         [TSLA]
2022-12-29        [TSLA]           [MU]
2022-12-30        [TSLA]           [MU]
2023-01-03        [TSLA]           [MU]
2023-01-04        [TSLA]           [MU]
2023-01-05        [TSLA]           [MU]
2023-01-06        [TSLA]           [MU]
2023-01-09        [TSLA]           [MU]
2023-01-10        [TSLA]           [MU]
2023-01-11        [TSLA]           [MU]
2023-01-12        [TSLA]         [GOOG]
2023-01-13        [TSLA]          [SPY]
2023-01-17        [TSLA]          [SPY]
2023-01-18          [MU]         [TSLA]
2023-01-19         [SPY]         [TSLA]
2023-01-20         [SPY]         [GOOG]


In [16]:
test_results.columns

In [17]:
# Sweep confidence thresholds and compare models on the high-conviction subsets
thresholds = [0.50, 0.52, 0.54, 0.56, 0.58, 0.60]

print("=========================================================================")
print("CONFIDENCE THRESHOLD SWEEP (LSTM CONVICTION SUBSETS)")
print("=========================================================================")
print(f"{'Threshold':<10} | {'Coverage':<8} | {'Sample Count':<12} | {'Baseline Acc':<12} | {'LogReg Acc':<10} | {'RF Acc':<8} | {'LSTM Acc':<8}")
print("-" * 88)

for t in thresholds:
    # Filter the test set for high conviction predictions
    # We find rows where LSTM's Outperform_Prob >= t (confident buy) or Outperform_Prob <= (1 - t) (confident sell)
    mask = (test_results['Outperform_Prob'] >= t) | (test_results['Outperform_Prob'] <= (1 - t))
    subset = test_results[mask]
    
    # Calculate coverage and count
    count = len(subset)
    coverage = count / len(test_results)
    
    if count == 0:
        print(f"{t:<10.2f} | {coverage:<8.2%} | {count:<12d} | {'N/A':<12} | {'N/A':<10} | {'N/A':<8} | {'N/A':<8}")
        continue
        
    # Evaluate Dummy Classifier (Majority Class on this subset)
    val_counts = subset['Target'].value_counts()
    majority_class_acc = val_counts.max() / count if len(val_counts) > 0 else 0.0
    
    # Evaluate other models on the exact same high-conviction indices
    lr_acc = accuracy_score(subset['Target'], lr_pred[mask])
    rf_acc = accuracy_score(subset['Target'], rf_pred[mask])
    lstm_acc = accuracy_score(subset['Target'], lstm_pred[mask])
    
    print(f"{t:<10.2f} | {coverage:<8.2%} | {count:<12d} | {majority_class_acc:<12.4f} | {lr_acc:<10.4f} | {rf_acc:<8.4f} | {lstm_acc:<8.4f}")


CONFIDENCE THRESHOLD SWEEP (LSTM CONVICTION SUBSETS)
Threshold  | Coverage | Sample Count | Baseline Acc | LogReg Acc | RF Acc   | LSTM Acc
----------------------------------------------------------------------------------------
0.50       | 100.00%  | 3520         | 0.5000       | 0.4920     | 0.5000   | 0.5006  
0.52       | 98.35%   | 3462         | 0.5014       | 0.4922     | 0.5014   | 0.5012  
0.54       | 81.85%   | 2881         | 0.5005       | 0.4918     | 0.5005   | 0.5005  
0.56       | 3.52%    | 124          | 0.5645       | 0.4032     | 0.4355   | 0.4355  
0.58       | 0.03%    | 1            | 1.0000       | 0.0000     | 0.0000   | 0.0000  
0.60       | 0.00%    | 0            | N/A          | N/A        | N/A      | N/A     


In [18]:
# Let's evaluate each model's top-ranked and bottom-ranked picks (k=1) on each day
# This measures their accuracy strictly on their highest-conviction choices relative to other stocks

# Extract predicted probabilities for all models
# For Logistic Regression and Random Forest, we can get probabilities using predict_proba
lr_probs = lr.predict_proba(X_test_flat)[:, 1]
rf_probs = rf.predict_proba(X_test_flat)[:, 1]
lstm_probs_outperform = lstm_probs[:, 1]

# Assign probabilities to temporary evaluation dataframes
eval_df = test_results.copy()
eval_df['LR_Prob'] = lr_probs
eval_df['RF_Prob'] = rf_probs
eval_df['LSTM_Prob'] = lstm_probs_outperform

def compute_ranking_accuracy(df, prob_col):
    correct_long = 0
    correct_short = 0
    total_days = 0
    
    for timestamp, group in df.groupby('timestamp'):
        if len(group) < 2:
            continue
        total_days += 1
        
        # Sort by the model's predicted probability
        sorted_group = group.sort_values(by=prob_col, ascending=False)
        
        # Top 1 (Long)
        top_1 = sorted_group.iloc[0]
        if top_1['Target'] == 1:
            correct_long += 1
            
        # Bottom 1 (Short)
        bottom_1 = sorted_group.iloc[-1]
        if bottom_1['Target'] == 0:
            correct_short += 1
            
    long_acc = correct_long / total_days
    short_acc = correct_short / total_days
    overall_acc = (correct_long + correct_short) / (2 * total_days)
    return long_acc, short_acc, overall_acc

print("=========================================================================")
print("RANKED PORTFOLIO ACCURACY (TOP-1 LONG & BOTTOM-1 SHORT)")
print("=========================================================================")
for model_name, prob_col in [("Logistic Regression", "LR_Prob"), ("Random Forest", "RF_Prob"), ("LSTM Classifier", "LSTM_Prob")]:
    long_acc, short_acc, overall_acc = compute_ranking_accuracy(eval_df, prob_col)
    print(f"{model_name:<25} : Long Acc = {long_acc:.2%} | Short Acc = {short_acc:.2%} | Overall Acc = {overall_acc:.2%}")


RANKED PORTFOLIO ACCURACY (TOP-1 LONG & BOTTOM-1 SHORT)
Logistic Regression       : Long Acc = 49.43% | Short Acc = 49.89% | Overall Acc = 49.66%
Random Forest             : Long Acc = 50.23% | Short Acc = 50.80% | Overall Acc = 50.51%
LSTM Classifier           : Long Acc = 47.27% | Short Acc = 49.43% | Overall Acc = 48.35%
